<a href="https://colab.research.google.com/github/albiecr/IxV-curve-analysis/blob/main/atividade_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) Escreva uma função que receba uma lista de números e retorne outra lista
com os números ímpares.

In [37]:
# Número ímpar não é divisível por 2 (deixa resto)

def filtrar_impares(numeros):
  return [n for n in numeros if n % 2 != 0]

# Exemplo
print(filtrar_impares([47, 52, 33, 75, 91, 12, 50]))

[47, 33, 75, 91]


2. Escreva uma função que receba uma lista de números e retorne outra lista
com os números primos presentes.


In [40]:
# Número primo só é divisível por 1 e por ele mesmo

def num_primo(n):
  if n < 2:
    return False
  if n == 2:
    return True
  if n % 2 == 0:
    return False
  limite = int(n**0.5) + 1
  for i in range(3, limite, 2):
    if n % i == 0:
      return False
  return True

def filtrar_primos(numeros):
  return [n for n in numeros if num_primo(n)]

# Exemplo
print(filtrar_primos([3331, 4586, 3341, 5471, 1009, 551, 1111]))

[3331, 5471, 1009]


3) Escreva uma função que receba duas listas e retorne outra lista com os
elementos que estão presentes em apenas uma das listas.


In [39]:
def apenas_em_uma(lista1, lista2):
  return [n for n in lista1 + lista2 if n not in lista1 or n not in lista2]

# Exemplo
print(apenas_em_uma([1, 2, 3, 4], [3, 4, 5, 6]))

[1, 2, 5, 6]


4. Dada uma lista de números inteiros, escreva uma função para encontrar o
segundo maior valor na lista.

In [41]:
def segundo_maior(lista):
    unicos = sorted(set(lista))
    if len(unicos) < 2:
        raise ValueError("A lista precisa ter pelo menos 2 valores distintos.")
    return unicos[-2]

# Exemplo
print(segundo_maior([4256, 9873, 1276, 3698]))

4256


5. Crie uma função que receba uma lista de tuplas, cada uma contendo o
nome e a idade de uma pessoa, e retorne a lista ordenada pelo nome das
pessoas em ordem alfabética.


In [42]:
def ordenar_por_nome(pessoas):
    return sorted(pessoas, key=lambda x: x[0].lower())

# Exemplo
pessoas = [("Lilien", 22), ("Gabriel", 19), ("Emanoela", 30)]
print(ordenar_por_nome(pessoas))

[('Emanoela', 30), ('Gabriel', 19), ('Lilien', 22)]


6. Como identificar e tratar outliers em uma coluna numérica usando desvio
padrão ou quartis?


In [43]:
# Método do desvio padrão

import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "idade": [18, 19, 20, 21, 22, 23, 120, 24, None, 25],
    "salario": [1500, 1600, 1550, 1700, 1650, 1580, 50000, None, 1620, 1590],
    "cidade": ["Fortaleza", "Fortaleza", "Sobral", "Sobral", None, "Fortaleza", "Sobral", "Caucaia", "Caucaia", "Fortaleza"]
})

col = "idade"  # <- coluna que você quer analisar

# calcula média e desvio ignorando NaN
media = df[col].mean()
std = df[col].std()

# z-score (vai dar NaN onde col é NaN)
z = (df[col] - media) / std

# identificar outliers
outliers = df[abs(z) > 3]

# remover outliers (mantém NaN como não-outlier)
df_sem_outliers = df[abs(z).fillna(0) <= 3]

# capar (winsorizar)
df[f"{col}_cap"] = df[col].clip(media - 3*std, media + 3*std)

print("Média:", media)
print("Std:", std)
print("\nOutliers:")
print(outliers)
print("\nDF com cap:")
print(df)


Média: 32.44444444444444
Std: 32.913185469926454

Outliers:
Empty DataFrame
Columns: [id, idade, salario, cidade]
Index: []

DF com cap:
   id  idade  salario     cidade  idade_cap
0   1   18.0   1500.0  Fortaleza       18.0
1   2   19.0   1600.0  Fortaleza       19.0
2   3   20.0   1550.0     Sobral       20.0
3   4   21.0   1700.0     Sobral       21.0
4   5   22.0   1650.0       None       22.0
5   6   23.0   1580.0  Fortaleza       23.0
6   7  120.0  50000.0     Sobral      120.0
7   8   24.0      NaN    Caucaia       24.0
8   9    NaN   1620.0    Caucaia        NaN
9  10   25.0   1590.0  Fortaleza       25.0


In [44]:
# Método dos Quartis

import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "idade": [18, 19, 20, 21, 22, 23, 120, 24, None, 25],
    "salario": [1500, 1600, 1550, 1700, 1650, 1580, 50000, None, 1620, 1590],
    "cidade": ["Fortaleza", "Fortaleza", "Sobral", "Sobral", None, "Fortaleza", "Sobral", "Caucaia", "Caucaia", "Fortaleza"]
})

col = "idade"

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

outliers_idade = df[(df[col] < lim_inf) | (df[col] > lim_sup)]

# Tratamento 1: remover outliers (mantém NaN como não-outlier)
df_idade_sem_outliers = df[~((df[col] < lim_inf) | (df[col] > lim_sup)) | (df[col].isna())]

# Tratamento 2: capar (winsorizar)
df[f"{col}_cap_iqr"] = df[col].clip(lim_inf, lim_sup)

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Limite inferior:", lim_inf, "Limite superior:", lim_sup)
print("\nOutliers (idade):")
print(outliers_idade)
print("\nDataFrame com idade capada (IQR):")
print(df)


Q1: 20.0 Q3: 24.0 IQR: 4.0
Limite inferior: 14.0 Limite superior: 30.0

Outliers (idade):
   id  idade  salario  cidade
6   7  120.0  50000.0  Sobral

DataFrame com idade capada (IQR):
   id  idade  salario     cidade  idade_cap_iqr
0   1   18.0   1500.0  Fortaleza           18.0
1   2   19.0   1600.0  Fortaleza           19.0
2   3   20.0   1550.0     Sobral           20.0
3   4   21.0   1700.0     Sobral           21.0
4   5   22.0   1650.0       None           22.0
5   6   23.0   1580.0  Fortaleza           23.0
6   7  120.0  50000.0     Sobral           30.0
7   8   24.0      NaN    Caucaia           24.0
8   9    NaN   1620.0    Caucaia            NaN
9  10   25.0   1590.0  Fortaleza           25.0


7. Como concatenar vários DataFrames (empilhando linhas ou colunas),
mesmo que tenham colunas diferentes?
Dica: Utiliza-se pd.concat() especificando axis=0 (linhas) ou axis=1
(colunas). Quando há colunas diferentes, os valores ausentes são
preenchidos com NaN.

In [45]:
import pandas as pd

df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "nome": ["Ana", "Bruno", "Carlos"],
    "idade": [19, 25, 30]
})

df2 = pd.DataFrame({
    "id": [4, 5],
    "nome": ["Duda", "Edu"],
    "salario": [2500, 3200],   # coluna diferente (não existe no df1)
    "cidade": ["Fortaleza", "Sobral"]  # coluna diferente (não existe no df1)
})

df1, df2

# Concatenar por linhas
df_linhas = pd.concat([df1, df2], axis=0, ignore_index=True)
df_linhas

# Concatenar por colunas
df_colunas = pd.concat([df1, df2], axis=1)
df_colunas

# Valores ausentes preenchidos com NaN
df_linhas_fill = df_linhas.fillna({
    "idade": 0,
    "salario": 0,
    "cidade": "Desconhecida"
})
df_linhas_fill


,id,nome,idade,salario,cidade
0,1,Ana,19.0,0.0,Desconhecida
1,2,Bruno,25.0,0.0,Desconhecida
2,3,Carlos,30.0,0.0,Desconhecida
3,4,Duda,0.0,2500.0,Fortaleza
4,5,Edu,0.0,3200.0,Sobral


8. Utilizando pandas, como realizar a leitura de um arquivo CSV em um
DataFrame e exibir as primeiras linhas?


In [46]:
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# Exibe as primeiras 5 linhas
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


9. Utilizando pandas, como selecionar uma coluna específica e filtrar linhas
em um “DataFrame” com base em uma condição?


In [47]:
import pandas as pd
from IPython.display import display

# Ler o CSV do Titanic
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# Selecionar uma coluna
display(df["age"].head(10))

# Filtrar linhas por condição (ex.: idade >= 18)
display(df[df["age"] >= 18].head(10))


,age
0,22.0
1,38.0
2,26.0
3,35.0
4,35.0
5,NaN
6,54.0
7,2.0
8,27.0
9,14.0


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
12,0,3,male,20.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
13,0,3,male,39.0,1,5,31.2750,S,Third,man,True,NaN,Southampton,no,False


10. Utilizando pandas, como lidar com valores ausentes (NaN) em um
DataFrame?

In [48]:
import pandas as pd
from IPython.display import display

# Ler o CSV do Titanic
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# 1) Identificar valores ausentes (NaN) por coluna
display(df.isna().sum().sort_values(ascending=False))

# 2) Remover linhas com NaN (ex.: remover apenas se age estiver ausente)
df_sem_nan_age = df.dropna(subset=["age"])
display(df_sem_nan_age.head(10))

# 3) Preencher NaN (ex.: preencher age com a mediana e deck com "Unknown")
df_preenchido = df.copy()
df_preenchido["age"] = df_preenchido["age"].fillna(df_preenchido["age"].median())
df_preenchido["deck"] = df_preenchido["deck"].fillna("Unknown")
display(df_preenchido.head(10))

,0
deck,688
age,177
embarked,2
embark_town,2
sex,0
pclass,0
survived,0
fare,0
parch,0
sibsp,0


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False
10,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Unknown,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Unknown,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Unknown,Southampton,no,True
5,0,3,male,28.0,0,0,8.4583,Q,Third,man,True,Unknown,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,Unknown,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,Unknown,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,Unknown,Cherbourg,yes,False
